## PhishTank Feature Extraction
This section presents the extraction of URL-based features from the PhishTank dataset. To maintain consistency with the reduced UCI feature set, only 10 stable and reproducible URL-based features were selected. These features were chosen because they can be derived directly from the URL string without relying on external services such as WHOIS, DNS queries, or webpage-content scraping. This makes the extraction process more reliable, computationally efficient, and suitable for large-scale real-world analysis.

In [103]:
# Importing required libraries
import pandas as pd   
import numpy as np   
import re             # for pattern matching (used in URL feature extraction)
from urllib.parse import urlparse   # to break URL into components like domain, path
import ipaddress as ip   # to check if URL contains an IP address
import warnings
warnings.filterwarnings('ignore')   # to suppress unnecessary warning messages

In [202]:
# Loading the PhishTank dataset from the CSV file
# This dataset contains phishing URLs which will be used for feature extraction
phishtank_df = pd.read_csv('Phishing_URL.csv')
# Displaying first few rows to understand the structure
phishtank_df.head()

,phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target
0,9360218,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,http://www.phishtank.com/phish_detail.php?phis...,2026-03-05T21:10:25+00:00,yes,2026-03-05T21:12:49+00:00,yes,Other
1,9360217,https://formulaire-livraison.com/,http://www.phishtank.com/phish_detail.php?phis...,2026-03-05T21:01:35+00:00,yes,2026-03-05T21:03:21+00:00,yes,Other
2,9360216,https://formulaire-livraison.com/captcha/captc...,http://www.phishtank.com/phish_detail.php?phis...,2026-03-05T21:01:21+00:00,yes,2026-03-05T21:03:21+00:00,yes,Other
3,9360215,https://sso-ndax-logi.webflow.io/,http://www.phishtank.com/phish_detail.php?phis...,2026-03-05T21:01:02+00:00,yes,2026-03-05T21:03:21+00:00,yes,Other
4,9360213,https://atolimba.vercel.app/,http://www.phishtank.com/phish_detail.php?phis...,2026-03-05T20:53:51+00:00,yes,2026-03-05T21:03:21+00:00,yes,Other


In [107]:
# Checking the shape of the dataset
print("Dataset shape:", phishtank_df.shape)

Dataset shape: (56094, 8)


In [109]:
# Checking column names to understand what information is available in the dataset
print(phishtank_df.columns)

Index(['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified',
       'verification_time', 'online', 'target'],
      dtype='object')


# Dataset Column Selection
The PhishTank dataset contains several columns such as phish_detail_url, url, submission_time, verification_time, online, and target. However, most of these columns represent metadata related to phishing reports rather than the actual characteristics of the website itself.
For the purpose of phishing detection using machine learning, the url column is the most relevant, as it contains structural and lexical information that can help distinguish phishing websites from legitimate ones. Therefore, only the url field is selected for further processing.
From each URL, a set of lexical features is extracted, including URL length, presence of special characters, subdomains, and other commonly used indicators in phishing detection research.
In addition, a Result column is created, where all entries are assigned a value of -1 to represent phishing instances. This follows the same labeling convention used in the UCI Phishing Websites benchmark dataset, ensuring consistency across datasets.

In [112]:
# Selecting only the URL column from the dataset
# The other columns contain metadata and are not needed for feature extraction
df = phishtank_df[['url']].copy()
# Creating the target column
# Since all URLs in PhishTank are phishing URLs, assigning label -1 to all entries
# This keeps the same labeling format as the UCI dataset
df['Result'] = -1
# Displaying first few rows to check the changes
df.head()



,url,Result
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1
1,https://formulaire-livraison.com/,-1
2,https://formulaire-livraison.com/captcha/captc...,-1
3,https://sso-ndax-logi.webflow.io/,-1
4,https://atolimba.vercel.app/,-1


In [114]:
# Checking for missing values in the dataset
df.isnull().sum()
print("Missing values in each column:\n", df.isnull().sum())

Missing values in each column:
 url       0
Result    0
dtype: int64


In [116]:
df.shape

(56094, 2)

In [118]:
# Checking for duplicate URLs in the dataset
# Duplicate URLs can create redundancy and may affect model performance
duplicate_url = df['url'].duplicated().sum()
# Printing the number of duplicate URLs found
print(f"Number of duplicate URLs: {duplicate_url}")
# If duplicates are present, they will be removed in the next step to ensure that each URL appears only once in the dataset

Number of duplicate URLs: 7


In [120]:
# Removing duplicate URLs from the dataset
# This helps remove repeated entries and improves overall data quality
df = df.drop_duplicates(subset=['url'])
# Resetting the index after removing duplicates
# This keeps the index clean and properly ordered
df = df.reset_index(drop=True)
# Displaying the shape after removing duplicates
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (56087, 2)


In [122]:
# Installing tldextract (only needed if not already installed)
import sys
!{sys.executable} -m pip install tldextract
import tldextract

In [123]:
# Selecting only the URL column
df = phishtank_df[['url']].copy()
# Creating the target column
df['Result'] = -1
# Removing duplicate URLs
df = df.drop_duplicates(subset=['url'])
df = df.reset_index(drop=True)

# Feature 1: having_IP_Address
def having_IP_Address(url):
    try:
        hostname = urlparse(url).hostname

        # If hostname cannot be extracted, treat as non-IP
        if hostname is None:
            return 1

        # Check whether the hostname is an IP address
        ip.ip_address(hostname)
        return -1

    except ValueError:
        return 1

df['having_IP_Address'] = df['url'].apply(having_IP_Address)
print(df.shape)

(56087, 3)


In [126]:
df.head()

,url,Result,having_IP_Address
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1
1,https://formulaire-livraison.com/,-1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1
3,https://sso-ndax-logi.webflow.io/,-1,1
4,https://atolimba.vercel.app/,-1,1


In [128]:
df.shape

(56087, 3)

In [130]:
print(df['having_IP_Address'].value_counts())

having_IP_Address
 1    55951
-1      136
Name: count, dtype: int64


In [132]:
df['having_IP_Address'].unique()

array([ 1, -1])

In [134]:
# Feature 2: URL_Length
# This feature measures the length of the URL
# Longer URLs are often used in phishing attacks to hide suspicious or misleading content

# 1  = short URL (likely legitimate)
# 0  = medium-length URL (potentially suspicious)
# -1 = long URL (more likely phishing)

def URL_Length(url):
    if len(url) < 54:
        return 1
    elif len(url) <= 75:
        return 0
    else:
        return -1

# Applying the feature extraction
df['URL_Length'] = df['url'].apply(URL_Length)

In [136]:
df.head()

,url,Result,having_IP_Address,URL_Length
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1
1,https://formulaire-livraison.com/,-1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1
4,https://atolimba.vercel.app/,-1,1,1


In [138]:
print(df['URL_Length'].value_counts())

URL_Length
 1    39043
-1    11797
 0     5247
Name: count, dtype: int64


In [140]:
# Feature 3: Shortining_Service
# Checks if the URL uses a URL shortening service
# Phishing websites often use shorteners to hide the actual destination of the link
def Shortining_Service(url):
    shorteners = r"bit\.ly|goo\.gl|tinyurl\.com|ow\.ly|t\.co|is\.gd|buff\.ly|adf\.ly|bit\.do" 
    return -1 if re.search(shorteners, url, re.IGNORECASE) else 1
# Applying the feature
df['Shortining_Service'] = df['url'].apply(Shortining_Service)


In [142]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1
1,https://formulaire-livraison.com/,-1,1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1
4,https://atolimba.vercel.app/,-1,1,1,1


In [144]:
print(df['Shortining_Service'].value_counts())

Shortining_Service
 1    52466
-1     3621
Name: count, dtype: int64


In [146]:
# Feature 4: having_At_Symbol
# Checks if the URL contains @ symbol
# Browsers ignore everything before @, which can be used to mislead users
def having_At_Symbol(url):
    return -1 if '@' in url else 1
# Applying the feature
df['having_At_Symbol'] = df['url'].apply(having_At_Symbol)

In [148]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1
4,https://atolimba.vercel.app/,-1,1,1,1,1


In [150]:
print(df['having_At_Symbol'].value_counts())

having_At_Symbol
 1    55596
-1      491
Name: count, dtype: int64


In [152]:
def double_slash_redirecting(url):
    try:
        # Remove protocol part
        protocol_removed = url.split('//', 1)[-1]
        
        # Check if '//' appears again
        if '//' in protocol_removed:
            return -1  # suspicious
        else:
            return 1   # normal
    except:
        return 1
df['double_slash_redirecting'] = df['url'].apply(double_slash_redirecting)

In [154]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1,1
4,https://atolimba.vercel.app/,-1,1,1,1,1,1


In [156]:
print(df['double_slash_redirecting'].value_counts())

double_slash_redirecting
 1    55359
-1      728
Name: count, dtype: int64


In [158]:
# Feature 6: Prefix_Suffix
# This feature checks whether the domain contains a hyphen (-)
# Phishing websites often use hyphens to mimic legitimate domains (e.g., secure-paypal.com)
# -1 = hyphen present (suspicious)
#  1 = no hyphen
def Prefix_Suffix(url):
    try:
        hostname = urlparse(url).hostname
        return -1 if '-' in hostname else 1
    except:
        return 1
# Applying the feature
df['Prefix_Suffix'] = df['url'].apply(Prefix_Suffix)

In [160]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1,1,-1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1,1,-1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1,1,-1
4,https://atolimba.vercel.app/,-1,1,1,1,1,1,1


In [162]:
df['Prefix_Suffix'].unique()

array([ 1, -1])

In [164]:
print(df['Prefix_Suffix'].value_counts())

Prefix_Suffix
 1    40181
-1    15906
Name: count, dtype: int64


In [166]:
# Feature 7: having_Sub_Domain
# This feature checks the number of subdomains in the URL
# Phishing websites often use multiple subdomains to appear more legitimate
#  1= no subdomain or only 'www' (likely legitimate)
#  0 = one subdomain (suspicious)
# -1 = multiple subdomains (more likely phishing)
def having_Sub_Domain(url):
    try:
        ext = tldextract.extract(url)
        subdomain = ext.subdomain

        if subdomain == '' or subdomain == 'www':
            return 1
        
        # Count number of subdomain levels
        num_subdomains = subdomain.count('.') + 1
        
        if num_subdomains == 1:
            return 0
        else:
            return -1

    except:
        return 1
# Applying the feature
df['having_Sub_Domain'] = df['url'].apply(having_Sub_Domain)

In [168]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1,1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1,1,-1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1,1,-1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1,1,-1,0
4,https://atolimba.vercel.app/,-1,1,1,1,1,1,1,0


In [170]:
print(df['having_Sub_Domain'].value_counts())

having_Sub_Domain
 0    33824
 1    19102
-1     3161
Name: count, dtype: int64


In [172]:
# Feature 8: HTTPS_token
# This feature checks if the word 'https' appears in the URL after removing the protocol
# Phishing URLs may include 'https' in other parts to look secure
def HTTPS_token(url):
    try:
        url = url.lower()
        
        # Remove protocol
        if url.startswith('https://'):
            url = url[8:]
        elif url.startswith('http://'):
            url = url[7:]
        
        return -1 if 'https' in url else 1
    except:
        return 1

# Applying the feature
df['HTTPS_token'] = df['url'].apply(HTTPS_token)

In [174]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,HTTPS_token
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1,1,1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1,1,-1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1,1,-1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1,1,-1,0,1
4,https://atolimba.vercel.app/,-1,1,1,1,1,1,1,0,1


In [176]:
print(df['HTTPS_token'].value_counts())

HTTPS_token
 1    55186
-1      901
Name: count, dtype: int64


In [178]:
# Feature 9: port
# This feature checks whether the URL uses a non-standard port number
# Standard ports are 80 (HTTP) and 443 (HTTPS)
# -1 = non-standard port (suspicious)
#  1 = no port or standard port

def port(url):
    try:
        parsed = urlparse(url)
        
        if parsed.port is None:
            return 1   # no port specified
        elif parsed.port in [80, 443]:
            return 1   # standard port
        else:
            return -1  # non-standard port
    except:
        return 1

# Applying the feature
df['port'] = df['url'].apply(port)

In [180]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,HTTPS_token,port
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1,1,1,1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1,1,-1,1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1,1,-1,1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1,1,-1,0,1,1
4,https://atolimba.vercel.app/,-1,1,1,1,1,1,1,0,1,1


In [188]:
print(df['port'].value_counts())

port
 1    56073
-1       14
Name: count, dtype: int64


In [190]:
# Feature 10: Abnormal_URL
# This feature checks whether the hostname is properly present in the URL
# Abnormal URLs may not follow proper structure
# -1 = abnormal URL
#  1 = normal URL

def Abnormal_URL(url):
    try:
        parsed = urlparse(url)
        return 1 if parsed.hostname else -1
    except:
        return -1

# Applying the feature
df['Abnormal_URL'] = df['url'].apply(Abnormal_URL)

In [192]:
df.head()

,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,HTTPS_token,port,Abnormal_URL
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1,1,1,1,1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1,1,-1,1,1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1,1,-1,1,1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1,1,-1,0,1,1,1
4,https://atolimba.vercel.app/,-1,1,1,1,1,1,1,0,1,1,1


In [194]:
print(df['Abnormal_URL'].value_counts())

Abnormal_URL
1    56087
Name: count, dtype: int64


In [196]:
df.columns

Index(['url', 'Result', 'having_IP_Address', 'URL_Length',
       'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting',
       'Prefix_Suffix', 'having_Sub_Domain', 'HTTPS_token', 'port',
       'Abnormal_URL'],
      dtype='object')

In [93]:
df.shape

(56087, 12)

In [95]:
# Final check
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
df.head()

Shape: (56087, 12)

Missing values:
 url                         0
Result                      0
having_IP_Address           0
URL_Length                  0
Shortining_Service          0
having_At_Symbol            0
double_slash_redirecting    0
Prefix_Suffix               0
having_Sub_Domain           0
HTTPS_token                 0
port                        0
Abnormal_URL                0
dtype: int64


,url,Result,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,HTTPS_token,port,Abnormal_URL
0,https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...,-1,1,-1,1,1,1,1,1,1,1,1
1,https://formulaire-livraison.com/,-1,1,1,1,1,1,-1,1,1,1,1
2,https://formulaire-livraison.com/captcha/captc...,-1,1,1,1,1,1,-1,1,1,1,1
3,https://sso-ndax-logi.webflow.io/,-1,1,1,1,1,1,-1,0,1,1,1
4,https://atolimba.vercel.app/,-1,1,1,1,1,1,1,0,1,1,1


In [97]:
print(df.head())
print(df.columns)

                                                 url  Result  \
0  https://ipfs.io/ipfs/bafkreiaocuu2mobhsyhx5lgu...      -1   
1                  https://formulaire-livraison.com/      -1   
2  https://formulaire-livraison.com/captcha/captc...      -1   
3                  https://sso-ndax-logi.webflow.io/      -1   
4                       https://atolimba.vercel.app/      -1   

   having_IP_Address  URL_Length  Shortining_Service  having_At_Symbol  \
0                  1          -1                   1                 1   
1                  1           1                   1                 1   
2                  1           1                   1                 1   
3                  1           1                   1                 1   
4                  1           1                   1                 1   

   double_slash_redirecting  Prefix_Suffix  having_Sub_Domain  HTTPS_token  \
0                         1              1                  1            1   
1             

# Feature Selection Justification
In this study, only 10 URL-based features were selected from the original UCI Phishing Websites dataset for use with the PhishTank dataset. The main reason for this selection was to ensure consistency between the training dataset (UCI) and the real-world validation dataset (PhishTank).
The selected features are lexical and URL-based, meaning they can be extracted directly from the URL string without relying on external services. Unlike features that depend on WHOIS lookups, DNS queries, or webpage content analysis, these features do not require real-time network calls. As a result, they are more computationally efficient, reliable, and suitable for large-scale extraction across the PhishTank dataset.
The PhishTank dataset initially contained 56,094 verified phishing URLs. After removing duplicate entries during preprocessing, the final dataset used for feature extraction consisted of 56,087 URLs.
During preliminary experimentation, network-dependent features such as domain registration length and age of domain were tested on a subset of URLs. However, these features frequently failed due to network errors, unavailable services, and incomplete registration records, making them unsuitable for large-scale implementation.
Similarly, several features from the UCI dataset, including Page Rank, Google Index, and web traffic, could not be extracted due to the lack of accessible external APIs and services.
Therefore, only stable and computationally efficient features were retained. This approach ensures that the model remains practical, scalable, and applicable to real-world phishing detection scenarios, where fast and reliable feature extraction is essential.

In [99]:
# Saving PhishTank features to CSV for cross-dataset validation
df.to_csv('PhishTank_features.csv', index=False)
print("Saved successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Saved successfully
Shape: (56087, 12)
Columns: ['url', 'Result', 'having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'HTTPS_token', 'port', 'Abnormal_URL']


In [101]:
# List of 10 URL-based features extracted in notebook
features_10 = [
    'having_IP_Address',
    'URL_Length',
    'Shortining_Service',
    'having_At_Symbol',
    'double_slash_redirecting',
    'Prefix_Suffix',
    'having_Sub_Domain',
    'HTTPS_token',
    'port',
    'Abnormal_URL']

# Select only the 10 features + Result
phish_clean = df[features_10 + ['Result']].copy()
# Optional: check shape and first few rows
print("Cleaned PhishTank dataset shape:", phish_clean.shape)
print(phish_clean.head())
# Save cleaned CSV without the url column
phish_clean.to_csv("PhishTank_10_features.csv", index=False)
print("Cleaned CSV saved successfully")

Cleaned PhishTank dataset shape: (56087, 11)
   having_IP_Address  URL_Length  Shortining_Service  having_At_Symbol  \
0                  1          -1                   1                 1   
1                  1           1                   1                 1   
2                  1           1                   1                 1   
3                  1           1                   1                 1   
4                  1           1                   1                 1   

   double_slash_redirecting  Prefix_Suffix  having_Sub_Domain  HTTPS_token  \
0                         1              1                  1            1   
1                         1             -1                  1            1   
2                         1             -1                  1            1   
3                         1             -1                  0            1   
4                         1              1                  0            1   

   port  Abnormal_URL  Result  
0     1  